# LangChain 应用开发中的模型实例化方式

主要目标：**在 LangChain 应用开发里，弄懂官方更推荐怎样创建模型实例？**

先说结论：
- 日常开发，优先使用 `init_chat_model()`
- 需要 Provider 专有能力时，再使用 `ChatXxx(...)` 专属类
- 对接 OpenAI 兼容端点时，使用 `ChatOpenAI(base_url=...)`


## 一、LangChain 推荐的主路径：`init_chat_model()`

这是 LangChain 官方推荐的统一入口。

适合场景：
- 想统一不同 Provider 的写法
- 想减少 import
- 后续可能切换模型供应商
- 要和 LangChain 的链、Agent、工具调用统一配合


In [ ]:
# 示例1.1：最常用写法，显式指定 provider:model
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model(
    "deepseek:deepseek-v4-flash",
    temperature=0.7,
)

response = llm.invoke("你是什么模型？")
print(response.content)


In [ ]:
# 示例1.2：可自动推断 provider 时，可省略前缀
llm = init_chat_model(
    "deepseek-v4-flash",
    temperature=0,
)

response = llm.invoke("你好")
print(response.content)


In [ ]:
# 示例1.3：同样适用于本地 Ollama
llm_ollama = init_chat_model(
    "ollama:qwen2.5:32b",
    temperature=0.7,
)

print(llm_ollama.invoke("你是什么模型").content)


In [ ]:
# 示例1.4：通过 init_chat_model() 对接 OpenAI 兼容端点
import os
from dotenv import load_dotenv
load_dotenv()

llm_dashscope = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    temperature=0.7,
)

print(llm_dashscope.invoke("你是什么模型").content)


这个写法的本质是：**把 OpenAI 这一层当作兼容协议入口**。

适合场景：
- 平台提供的是 OpenAI 兼容端点
- 仍然希望继续使用 `init_chat_model()` 统一入口

注意：这种方式适合兼容接口场景；如果依赖 Provider 非标准字段，仍然优先使用对应专属类。


### 这一节的结论

如果你在写 LangChain 应用，大多数情况下优先记住这一句就够了：

```python
llm = init_chat_model("provider:model", ...)
```


## 二、什么时候改用 Provider 专属类

`init_chat_model()` 是主路径，但不是唯一方案。

当你需要 **Provider 特有参数或特有返回字段** 时，改用专属类更合适。

典型场景：
- DeepSeek 的 `reasoning_content` 等非标准字段
- Anthropic 的 `thinking`
- Ollama 的 `num_ctx`、`num_predict`


In [ ]:
# 示例2.1：DeepSeek 专属类
from langchain_deepseek import ChatDeepSeek

llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0.7,
    timeout=30,
    max_tokens=1024,
    max_retries=3,
)

print(llm_deepseek.invoke("你是什么模型？").content)


In [ ]:
# 示例2.2：Ollama 专属类
from langchain_ollama import ChatOllama

llm_ollama = ChatOllama(
    model="qwen2.5:32b",
    temperature=0.7,
    base_url="http://localhost:11434",
    # num_ctx=32768,
    # num_predict=2048,
)

print(llm_ollama.invoke("你是什么模型").content)


### 这一节的结论

- 默认优先 `init_chat_model()`
- 只有在你明确需要 Provider 专有能力时，再切到 `ChatXxx(...)`


## 三、什么时候使用 OpenAI 兼容接口

有些平台不是 LangChain 推荐的主路径，但它们提供了 OpenAI 兼容端点。

这时可以用 `ChatOpenAI(base_url=..., api_key=...)` 对接。

适合场景：
- 阿里云百炼
- 百度千帆的兼容端点
- 本地 vLLM / LM Studio

注意：如果你依赖非标准字段，不建议硬走兼容接口。


In [ ]:
# 示例3.1：通过 LangChain 对接阿里云百炼
from langchain_openai import ChatOpenAI
import os

llm_dashscope = ChatOpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    model="glm-5",
    temperature=0.7,
)

print(llm_dashscope.invoke("你是什么模型").content)


## 四、补充：运行时动态切换模型

这不是主干内容，但在多模型应用里很有用。

适合场景：
- 同一个应用按任务类型切不同模型
- 默认走本地或便宜模型
- 特殊请求临时切到更强模型


In [ ]:
# 示例4.1：运行时切换模型
llm_dynamic = init_chat_model(
    temperature=0,
    configurable_fields=("model",),
)

def ask(user_input: str):
    if "会议纪要" in user_input or "内部文档" in user_input:
        model_name = "ollama:qwen2.5:32b"
    else:
        model_name = "deepseek:deepseek-v4-pro"

    response = llm_dynamic.invoke(
        user_input,
        config={"configurable": {"model": model_name}},
    )
    print(f"使用模型: {model_name}")
    print(response.content)

ask("先告诉我你是什么模型，再把这段公司会议纪要整理成 3 条结论")


## 五、最终选型建议

优先顺序建议：

1. 日常 LangChain 开发：`init_chat_model()`
2. 需要 Provider 高级能力：`ChatXxx(...)`
3. 对接 OpenAI 兼容端点：`ChatOpenAI(base_url=...)`
4. 多模型应用再考虑运行时动态切换

如果你只记一句话：**LangChain 里优先学会 `init_chat_model()`，其他都是补充。**
